In [1]:
import pandas as pd
import numpy as np
import altair as alt
import theme

alt.themes.register('main_theme', theme.main_theme)
alt.themes.enable('main_theme')

alt.data_transformers.disable_max_rows()

/tmp/ipykernel_25820/1193958727.py:6: AltairDeprecationWarning: 
Deprecated since `altair=5.5.0`. Use altair.theme instead.
Most cases require only the following change:

    # Deprecated
    alt.themes.enable('quartz')

    # Updated
    alt.theme.enable('quartz')

If your code registers a theme, make the following change:

    # Deprecated
    def custom_theme():
        return {'height': 400, 'width': 700}
    alt.themes.register('theme_name', custom_theme)
    alt.themes.enable('theme_name')

    # Updated
    @alt.theme.register('theme_name', enable=True)
    def custom_theme():
        return alt.theme.ThemeConfig(
            {'height': 400, 'width': 700}
        )

See the updated User Guide for further details:
    https://altair-viz.github.io/user_guide/api.html#theme
    https://altair-viz.github.io/user_guide/customization.html#chart-themes
  alt.themes.register('main_theme', theme.main_theme)


DataTransformerRegistry.enable('default')

In [2]:
#Define antigenic regions based on ChimeraX select interface tool with clesrovimab precursor RB1 6ous strucutre and nirsevimab precursor 5udc 
#ChimeraX Sel Interface with default settings (less stringent) as these closely match published residues bound by these mAbs 
alt.renderers.set_embed_options(actions={"export": True, "source": False, "editor": False})

# -----------------------------
# Load data
df = pd.read_csv("293T-TIM1_entry_func_effects.csv")
df["site"] = pd.to_numeric(df["site"], errors="coerce")

# -----------------------------
# QC filters
df_filt = (
    df.dropna(subset=["site", "effect"])
      .query("times_seen >= 2")
      .query("effect_std <= 2")
      .copy()
)

# -----------------------------
# Define antigenic-site ranges for nirsevimab and clesrovimab
nirsevimab_ranges = [(63, 69), (83, 83), (196, 197), (200, 202), (204, 212), (216, 216), (295, 295)]
clesrovimab_ranges = [(426, 429), (431, 433), (440, 447), (461, 461), (463, 468), (470, 470)]

def in_any_range(site_series, ranges):
    mask = pd.Series(False, index=site_series.index)
    for lo, hi in ranges:
        mask |= site_series.between(lo, hi, inclusive="both")
    return mask

# Assign region labels
df_filt["region"] = np.nan
df_filt.loc[in_any_range(df_filt["site"], nirsevimab_ranges), "region"] = "Nirsevimab"
df_filt.loc[in_any_range(df_filt["site"], clesrovimab_ranges), "region"] = "Clesrovimab"

plot_df = df_filt.dropna(subset=["region"]).copy()

# -----------------------------
# Plot styling
order = ["Nirsevimab", "Clesrovimab"]
colors = {
    "Clesrovimab": "#6A51A3",   # purple
    "Nirsevimab": "#CB181D",  # red
}

# Add jitter
plot_df["jitter"] = np.random.normal(loc=0, scale=0.2, size=len(plot_df))

# -----------------------------
# Plot
points = alt.Chart(plot_df).mark_circle(
    size=40,
    opacity=0.6   # slightly less opaque than before
).encode(
    y=alt.Y("region:N", sort=order, title=None),
    x=alt.X("effect:Q", title=["Mutation effect on cell entry"]),
    yOffset=alt.YOffset("jitter:Q"),
    color=alt.Color(
        "region:N",
        scale=alt.Scale(domain=order, range=[colors[o] for o in order]),
        legend=None
    ),
    tooltip=["site:Q", "wildtype:N", "mutant:N", "effect:Q"]
)

median_line = alt.Chart(plot_df).mark_tick(
    color="black",
    thickness=3,
    size=25
).encode(
    y=alt.Y("region:N", sort=order),
    x="median(effect):Q"
)

vline = alt.Chart(pd.DataFrame({"x": [0]})).mark_rule(
    color="black",
    size=1.25,
    opacity=1.0,
    strokeDash=[6, 6]
).encode(x="x:Q")

chart = alt.layer(points, median_line, vline).properties(
    height=120,
    width=360,
    title=alt.TitleParams(
        text="Cell-entry effects in monoclonal antibody antigenic regions",
        anchor="middle",
        fontSize=16,
        fontWeight="bold",
    )
)

chart.display()

/tmp/ipykernel_25820/854400372.py:32: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Nirsevimab' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_filt.loc[in_any_range(df_filt["site"], nirsevimab_ranges), "region"] = "Nirsevimab"


alt.LayerChart(...)